In [ ]:
# =============================================================================
# Imports
# =============================================================================

import numpy as np
import torch
import pandas as pd

from scipy.interpolate import interp1d
from scipy.stats import rankdata, friedmanchisquare, norm

from pymoo.optimize import minimize
from pymoo.core.problem import Problem

from pymoo.algorithms.soo.nonconvex.cmaes import CMAES
from pymoo.algorithms.soo.nonconvex.de import DE
from pymoo.algorithms.soo.nonconvex.pso import PSO
from pymoo.algorithms.soo.nonconvex.nelder import NelderMead
from pymoo.algorithms.soo.nonconvex.g3pcx import G3PCX
from pymoo.algorithms.soo.nonconvex.es import ES
from pymoo.algorithms.soo.nonconvex.pattern import PatternSearch
from pymoo.algorithms.soo.nonconvex.nrbo import NRBO

from x_msg.construct_msg_landscape import MSGLandscape
from x_msg.sampling import sobol_sampling


# =============================================================================
# Experiment Configuration
# =============================================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

D = 2
NUM_GAUSSIANS = 100

BASE_SEED = 42
N_RUNS = 11
MAX_EVALS = 3000

DATA_PATH = r".\res_rq2\msg_ela\results_max_min_max_2d.pt"


# =============================================================================
# Utility Functions
# =============================================================================

def unique_theta_list(theta_list, device=None):

    unique_thetas = []

    for t in theta_list:

        if device is not None:
            t = t.to(device)

        unique_thetas.append(t)

    return unique_thetas


def extract_uniform_trajectory(res, max_evals=3000):

    evals = []
    best_F = []

    for entry in res.history:

        evals.append(entry.evaluator.n_eval)
        best_F.append(entry.opt.get("F")[0])

    if len(evals) > 0 and evals[0] > 0:

        evals.insert(0, 0)
        best_F.insert(0, best_F[0])

    evals = np.array(evals)
    best_F = np.array(best_F)

    evals_unique, idx = np.unique(evals, return_index=True)
    best_unique = best_F[idx]

    common = np.arange(0, max_evals + 1)

    interp = interp1d(
        evals_unique,
        best_unique,
        kind="previous",
        bounds_error=False,
        fill_value=(best_unique[0], best_unique[-1]),
    )

    return common, interp(common)


# =============================================================================
# pymoo Problem
# =============================================================================

class MSGProblem(Problem):

    def __init__(self, landscape, dim, device="cuda"):

        super().__init__(n_var=dim, xl=0.0, xu=1.0)

        self.landscape = landscape
        self.device = device

    def _evaluate(self, X, out, *args, **kwargs):

        X_tensor = torch.tensor(
            X,
            dtype=torch.float32,
            device=self.device
        )

        with torch.no_grad():
            y = self.landscape(X_tensor)

        if y.dim() == 1:
            y = y.unsqueeze(1)

        out["F"] = y.cpu().numpy()


# =============================================================================
# Algorithms
# =============================================================================

def get_algorithms():

    return {
        "CMA-ES": CMAES(),
        "DE": DE(),
        "PSO": PSO(),
        "Nelder-Mead": NelderMead(),
        "G3PCX": G3PCX(),
        "ES": ES(),
        "PatternSearch": PatternSearch(),
        "NRBO": NRBO(),
    }


# =============================================================================
# Statistical Tests
# =============================================================================

def friedman_test(rank_matrix, alg_names):

    stat, p = friedmanchisquare(*rank_matrix.T)

    print("\n========== Friedman Test ==========")
    print("Statistic:", stat)
    print("p-value :", p)

    if p < 0.05:
        print("Significant difference detected")
    else:
        print("No significant difference")

    return stat, p


def holm_posthoc(rank_matrix, alg_names):

    n_runs, k = rank_matrix.shape

    mean_ranks = np.mean(rank_matrix, axis=0)

    print("\n========== Mean Ranks ==========")

    for a, r in zip(alg_names, mean_ranks):
        print(f"{a:15s} {r:.4f}")

    best = np.argmin(mean_ranks)

    print("\nReference algorithm:", alg_names[best])

    comparisons = []

    for i in range(k):

        if i == best:
            continue

        diff = abs(mean_ranks[i] - mean_ranks[best])

        z = diff / np.sqrt(k*(k+1)/(6*n_runs))

        p = 2*(1-norm.cdf(abs(z)))

        comparisons.append((alg_names[i], p))

    comparisons.sort(key=lambda x: x[1])

    print("\n========== Holm Test ==========")

    m = len(comparisons)

    for i,(alg,p) in enumerate(comparisons):

        alpha = 0.05/(m-i)

        reject = p < alpha

        print(f"{alg:15s} p={p:.6f} alpha={alpha:.6f} reject={reject}")


# =============================================================================
# Main Experiment
# =============================================================================

def main():

    means = sobol_sampling(D, NUM_GAUSSIANS, device=DEVICE, seed=BASE_SEED)

    result = torch.load(DATA_PATH)

    theta_history = result.get("theta_history", [])

    unique_thetas = unique_theta_list(theta_history, device=DEVICE)

    alg_names = list(get_algorithms().keys())

    records = []

    rank_matrix = []

    with torch.no_grad():

        for theta_id, theta in enumerate(unique_thetas):

            print("\n===================================")
            print("Landscape", theta_id)
            print("===================================")

            landscape = MSGLandscape(means, theta)

            problem = MSGProblem(landscape, D, DEVICE)

            for run in range(N_RUNS):

                seed = BASE_SEED + run

                algorithms = get_algorithms()

                best_values = []

                print("\nRun:", run)

                for name, alg in algorithms.items():

                    res = minimize(
                        problem,
                        alg,
                        ("n_evals", MAX_EVALS),
                        seed=seed,
                        verbose=False
                    )

                    best = res.F[0]

                    print(f"{name:15s} best={best:.6f}")

                    best_values.append(best)

                best_values = np.array(best_values)

                ranks = rankdata(best_values, method="average")

                rank_matrix.append(ranks)

                for i,alg in enumerate(alg_names):

                    records.append({
                        "landscape":theta_id,
                        "seed":seed,
                        "algorithm":alg,
                        "best_value":best_values[i],
                        "rank":ranks[i]
                    })

    rank_matrix = np.array(rank_matrix)

    friedman_test(rank_matrix, alg_names)

    holm_posthoc(rank_matrix, alg_names)

    df = pd.DataFrame(records)

    df.to_csv("results_raw.csv", index=False)

    mean_rank = df.groupby("algorithm")["rank"].mean().reset_index()

    mean_rank.rename(columns={"rank":"mean_rank"}, inplace=True)

    mean_rank.to_csv("results_mean_rank.csv", index=False)

    print("\nSaved:")
    print("results_raw.csv")
    print("results_mean_rank.csv")


# =============================================================================

if __name__ == "__main__":

    main()